In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip /content/drive/Shareddrives/thesis/jpeg_dataset.zip -d /content/

Streaming output truncated to the last 5000 lines.
file #9874:  bad zipfile offset (lseek):  1813700608
file #9875:  bad zipfile offset (lseek):  1813848064
file #9876:  bad zipfile offset (lseek):  1814036480
file #9877:  bad zipfile offset (lseek):  1814249472
file #9878:  bad zipfile offset (lseek):  1814388736
file #9879:  bad zipfile offset (lseek):  1814519808
file #9880:  bad zipfile offset (lseek):  1814708224
file #9881:  bad zipfile offset (lseek):  1814921216
file #9882:  bad zipfile offset (lseek):  1815085056
file #9883:  bad zipfile offset (lseek):  1815248896
file #9884:  bad zipfile offset (lseek):  1815404544
file #9885:  bad zipfile offset (lseek):  1815617536
file #9886:  bad zipfile offset (lseek):  1815748608
file #9887:  bad zipfile offset (lseek):  1815945216
file #9888:  bad zipfile offset (lseek):  1816125440
file #9889:  bad zipfile offset (lseek):  1816272896
file #9890:  bad zipfile offset (lseek):  1816453120
file #9891:  bad zipfile offset (lseek):  181659

In [ ]:
%%writefile generate_gax.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
from PIL import Image
import random
from tqdm import tqdm
import argparse
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.benchmark = True

class BatchGenerator(nn.Module):
    def __init__(self, batch_size, img_size=(224, 224)):
        super(BatchGenerator, self).__init__()
        self.W = nn.Parameter(torch.zeros(size=(batch_size, 3) + img_size) + 1)
        self.b = nn.Parameter(torch.zeros(size=(batch_size, 3) + img_size) + 0.01)
        self.act = nn.Tanh()

    def forward(self, x):
        return self.act(self.W * x + self.b)

def load_trained_resnet(model_path, device):
    model = models.resnet34(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    checkpoint = torch.load(model_path, map_location=device)
    state_dict = checkpoint.get('net', checkpoint.get('model', checkpoint.get('state_dict', checkpoint)))
    new_state_dict = {}
    for k, v in state_dict.items():
        if k == 'iter' or not isinstance(v, torch.Tensor): continue
        name = k.replace("backbone.", "")
        if name in model.state_dict() and v.shape == model.state_dict()[name].shape:
            new_state_dict[name] = v
    model.load_state_dict(new_state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model

class XRayDataset(Dataset):
    def __init__(self, img_files, data_dir, img_size=(224, 224)):
        self.img_files = img_files
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img_path = os.path.join(self.data_dir, img_name)
        img_pil = Image.open(img_path).convert('RGB').resize(self.img_size)
        x_np = np.array(img_pil).transpose(2, 0, 1) / 255.0
        return torch.from_numpy(x_np).float(), img_name

def run_batch_gax(model_path, output_dir, batch_size=16):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    BASE_DATA_DIR = "/content/jpeg_dataset/test"
    OUTPUT_DIR = output_dir
    N_ITER = 150
    LR = 0.1
    IMG_SIZE = (224, 224)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Loading ResNet34 on {device}...")
    model = load_trained_resnet(model_path, device)

    class_folders = [f for f in os.listdir(BASE_DATA_DIR) if os.path.isdir(os.path.join(BASE_DATA_DIR, f))]

    for class_name in class_folders:
        data_dir = os.path.join(BASE_DATA_DIR, class_name)
        target_label = 1 if class_name.lower() == 'pneumonia' else 0

        all_img_files = [f for f in os.listdir(data_dir) if f.endswith(('.jpeg', '.jpg', '.png'))]

        img_files = []
        for f in all_img_files:
            base_name = f"op.{f}.test.mult"
            if not (os.path.exists(os.path.join(OUTPUT_DIR, f"{base_name}.npy")) and
                    os.path.exists(os.path.join(OUTPUT_DIR, f"{base_name}.COS.npy"))):
                img_files.append(f)

        print(f"\nProcessing Class: {class_name} | Target Label: {target_label}")
        print(f"Found {len(all_img_files)} images total. {len(img_files)} remaining to process.")

        if len(img_files) == 0:
            continue

        dataset = XRayDataset(img_files, data_dir, IMG_SIZE)
        # Re-enabled pin_memory and higher workers for the Colab GPU
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

        for batch_x, batch_names in tqdm(dataloader, desc=f"Generating Batched GAX ({class_name})"):
            actual_batch_size = batch_x.size(0)
            batch_x = batch_x.to(device, non_blocking=True)

            with torch.no_grad():
                base_logits = model(batch_x)
                base_probs = F.softmax(base_logits, dim=1)
                n_class = base_probs.shape[1]

            netG = BatchGenerator(batch_size=actual_batch_size, img_size=IMG_SIZE).to(device)
            optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(0.9, 0.999))

            score_constants = torch.zeros((actual_batch_size, n_class), device=device) - (1.0 / (n_class - 1))
            score_constants[:, target_label] = 1.0

            batch_imgs_history = [[] for _ in range(actual_batch_size)]
            batch_scores_history = [[] for _ in range(actual_batch_size)]

            epsilon = 1e-4
            similarity_loss_factor = 1.0

            for i in range(N_ITER):
                netG.train()
                optimizerG.zero_grad()

                attr_op = netG(batch_x)
                x_aug = batch_x * attr_op

                aug_logits = model(x_aug)
                aug_probs = F.softmax(aug_logits, dim=1)

                co_score_tensor = (aug_probs - base_probs) * score_constants
                co_scores = torch.sum(co_score_tensor, dim=1)

                sim_losses = similarity_loss_factor / torch.mean((attr_op - batch_x + epsilon)**2 / (batch_x + epsilon), dim=[1,2,3])

                losses = -co_scores + sim_losses
                loss = losses.sum()

                loss.backward()
                optimizerG.step()

                netG.eval()
                with torch.no_grad():
                    current_masks = netG(batch_x).detach().cpu()
                    current_scores = co_scores.detach().cpu()

                    for b in range(actual_batch_size):
                        batch_scores_history[b].append(current_scores[b].item())
                        batch_imgs_history[b].append(current_masks[b].numpy().transpose(1, 2, 0))

            for b in range(actual_batch_size):
                img_name = batch_names[b]
                base_name = f"op.{img_name}.test.mult"
                mask_save_path = os.path.join(OUTPUT_DIR, f"{base_name}.npy")
                score_save_path = os.path.join(OUTPUT_DIR, f"{base_name}.COS.npy")

                np.save(mask_save_path, np.array(batch_imgs_history[b]))
                np.save(score_save_path, np.array(batch_scores_history[b]))

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate GAX masks for a given model")
    parser.add_argument("--model_path", type=str, default="/content/drive/Shareddrives/thesis/checkpointsbest_resnet34_v3.pth", help="Path to the model checkpoint")
    parser.add_argument("--output_dir", type=str, default="/content/drive/Shareddrives/thesis/resnet34_v4_gax", help="Directory to save the GAX results")
    parser.add_argument("--batch_size", type=int, default=16, help="Number of images to process simultaneously")
    args = parser.parse_args()

    run_batch_gax(args.model_path, args.output_dir, args.batch_size)

Writing generate_gax.py


note: upload compute_cheating_score.py and change mask_dir to  MASK_DIR = "/content/drive/Shareddrives/thesis/masked_dataset/test"


# Gax Generation and Cheating Score Evaluation for ResNet34_v2

##GAX Generation

In [ ]:
# Execution for v2
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v2.pth --output_dir /content/drive/Shareddrives/v2_gax_outputs/outputs

Traceback (most recent call last):
  File "/content/generate_gax.py", line 2, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2319, in <module>
    from torch import quantization as quantization  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/quantization/__init__.py", line 2, in <module>
    from .fake_quantize import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/quantization/fake_quantize.py", line 10, in <module>
    from torch.ao.quantization.fake_quantize import (
  File "/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/__init__.py", line 14, in <module>
    from .pt2e._numeric_debugger import (  # noqa: F401
  File "/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/pt2e/_numeric_debugger.py", line 8, in <module>
    from torch.ao.quantization.pt2e.graph_utils import bfs_trace

##Content Validation

In [ ]:
# count number of contents for each drive to validate
# contents validation for v2
# set folder path
import os
folder_path = '/content/drive/Shareddrives/v2_gax_outputs/outputs'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/v2_gax_outputs/outputs

Deep Count (Includes all sub-folders):
Total Files:   2388
Total Folders: 0
Grand Total:   2388


## Cheating Score

In [ ]:
# running cheating score on v2 while its files are in session storage so no need to download
# Define paths to match your v2 execution
script_path = "/content/drive/Shareddrives/thesis/compute_cheating_score.py"
gax_directory = "/content/drive/Shareddrives/v2_gax_outputs/outputs"
output_csv_path = "/content/drive/Shareddrives/v2_gax_outputs/cheating_scores_v2.csv"


# Run the script
!python "{script_path}" \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v2"

Starting Cheating Score calculation for model: resnet34_v2...
Scoring normal: 100% 885/885 [47:57<00:00,  3.25s/it]
Scoring pneumonia: 100% 602/602 [45:12<00:00,  4.51s/it]

Finished! Results saved to /content/drive/Shareddrives/v2_gax_outputs/cheating_scores_v2.csv

--- Final Summary (resnet34_v2) ---
Total Images Scored: 1487
  - normal: 885 images
  - pneumonia: 602 images
Shortcut Learning Detected: 1395 images (93.8%)
  - normal: 823 images
  - pneumonia: 572 images
Average Model Cheating Score: 0.7426 (74.3%)
Summary metrics successfully saved to /content/drive/Shareddrives/v2_gax_outputs/summary_cheating_scores.csv


## Download Cheating Scores

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/v2_gax_outputs/cheating_scores_v2.csv"
summary_csv_path = "/content/drive/Shareddrives/v2_gax_outputs/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

Found: cheating_scores_v2.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: summary_cheating_scores.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# GAX Generation and Cheating Score Evaluation for ResNet34_v4

## GAX Generation

In [ ]:
# Execution for v3 was done locally
# Execution for v4
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v4.pth --output_dir /content/drive/Shareddrives/thesis/resnet34_v4_gax

Loading ResNet34 on cuda...

Processing Class: pneumonia | Target Label: 1
Found 602 images total. 480 remaining to process.
Generating Batched GAX (pneumonia): 100% 30/30 [13:21<00:00, 26.70s/it]

Processing Class: normal | Target Label: 0
Found 885 images total. 0 remaining to process.


##Content Validation

In [ ]:
# count number of contents for each drive to validate
# contents validation for v4
# set folder path
import os
folder_path = '/content/drive/Shareddrives/thesis/resnet34_v4_gax'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/thesis/resnet34_v4_gax

Deep Count (Includes all sub-folders):
Total Files:   2053
Total Folders: 0
Grand Total:   2053


## Cheating Score

In [ ]:
# running cheating score on v4 while its files are in session storage so no need to download
# Define paths to match your v4 execution
script_path = "/content/drive/Shareddrives/thesis/compute_cheating_score.py"
gax_directory = "/content/drive/Shareddrives/thesis/resnet34_v4_gax"
output_csv_path = "/content/drive/Shareddrives/thesis/resnet34_v4_gax/cheating_scores_v4.csv"


# Run the script
!python "{script_path}" \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v4"

Starting Cheating Score calculation for model: resnet34_v4...
Scoring normal: 100% 885/885 [1:25:33<00:00,  5.80s/it]
Scoring pneumonia: 100% 602/602 [19:22<00:00,  1.93s/it]

Finished! Results saved to /content/drive/Shareddrives/thesis/resnet34_v4_gax/cheating_scores_v4.csv

--- Final Summary (resnet34_v4) ---
Total Images Scored: 1487
  - normal: 885 images
  - pneumonia: 602 images
Shortcut Learning Detected: 1361 images (91.5%)
  - normal: 789 images
  - pneumonia: 572 images
Average Model Cheating Score: 0.7230 (72.3%)
Summary metrics successfully saved to /content/drive/Shareddrives/thesis/resnet34_v4_gax/summary_cheating_scores.csv


## Cheating Score Download

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/thesis/resnet34_v4_gax/cheating_scores_v4.csv"
summary_csv_path = "/content/drive/Shareddrives/thesis/resnet34_v4_gax/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

Found: cheating_scores_v4.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: summary_cheating_scores.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# GAX Generation and Cheating Score Evaluation for ResNet34_v5

## GAX Generation

In [ ]:
# Execution for v5
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v5.pth --output_dir /content/drive/Shareddrives/v5_gax_outputs/resnet34_v5_gax

Loading ResNet34 on cuda...

Processing Class: pneumonia | Target Label: 1
Found 602 images total. 298 remaining to process.
Generating Batched GAX (pneumonia): 100% 19/19 [07:55<00:00, 25.04s/it]

Processing Class: normal | Target Label: 0
Found 885 images total. 0 remaining to process.


## Contents Validation

In [ ]:
# count number of contents for each drive to validate
# contents validation for v5
# set folder path
import os
folder_path = '/content/drive/Shareddrives/v5_gax_outputs/resnet34_v5_gax'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/v5_gax_outputs/resnet34_v5_gax

Deep Count (Includes all sub-folders):
Total Files:   2974
Total Folders: 0
Grand Total:   2974


##Cheating Score

In [ ]:
# running cheating score on v5 while its files are in session storage so no need to download
# Define paths to match your v5 execution
script_path = "/content/drive/Shareddrives/thesis/compute_cheating_score.py"
gax_directory = "/content/drive/Shareddrives/v5_gax_outputs/resnet34_v5_gax"
output_csv_path = "/content/drive/Shareddrives/v5_gax_outputs/cheating_scores_v5.csv"


# Run the script
!python "{script_path}" \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v5"

Starting Cheating Score calculation for model: resnet34_v5...
Scoring normal: 100% 885/885 [1:50:57<00:00,  7.52s/it]
Scoring pneumonia: 100% 602/602 [41:45<00:00,  4.16s/it]

Finished! Results saved to /content/drive/Shareddrives/v5_gax_outputs/cheating_scores_v5.csv

--- Final Summary (resnet34_v5) ---
Total Images Scored: 1487
  - normal: 885 images
  - pneumonia: 602 images
Shortcut Learning Detected: 1380 images (92.8%)
  - normal: 803 images
  - pneumonia: 577 images
Average Model Cheating Score: 0.7257 (72.6%)
Summary metrics successfully saved to /content/drive/Shareddrives/v5_gax_outputs/summary_cheating_scores.csv


## Download Cheating Score

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/v5_gax_outputs/cheating_scores_v5.csv"
summary_csv_path = "/content/drive/Shareddrives/v5_gax_outputs/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

Found: cheating_scores_v5.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: summary_cheating_scores.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# GAX Generation and Cheating Score Evaluation for ResNet34_v6

## GAX Generation

In [ ]:
# Execution for v6
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v6.pth --output_dir /content/drive/Shareddrives/v6_gax_outputs/outputs

Loading ResNet34 on cuda...

Processing Class: pneumonia | Target Label: 1
Found 602 images total. 0 remaining to process.

Processing Class: normal | Target Label: 0
Found 885 images total. 293 remaining to process.
Generating Batched GAX (normal): 100% 19/19 [08:23<00:00, 26.51s/it]


## Contents Validation

In [ ]:
# count number of contents for each drive to validate
# contents validation for v6, goal is 2974 files
# set folder path
import os
folder_path = '/content/drive/Shareddrives/v6_gax_outputs/outputs'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/v6_gax_outputs/outputs

Deep Count (Includes all sub-folders):
Total Files:   2974
Total Folders: 0
Grand Total:   2974


## Cheating Score

In [ ]:
# running cheating score on v6 while its files are in session storage so no need to download
# Define paths to match your v6 execution
script_path = "/content/drive/Shareddrives/thesis/compute_cheating_score.py"
gax_directory = "/content/drive/Shareddrives/v6_gax_outputs/outputs"
output_csv_path = "/content/drive/Shareddrives/v6_gax_outputs/cheating_scores_v6.csv"


# Run the script
!python "{script_path}" \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v6"

Starting Cheating Score calculation for model: resnet34_v6...
Scoring normal:  73% 647/885 [40:30<13:22,  3.37s/it]

## Download Cheating Score Results

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/v6_gax_outputs/cheating_scores_v6.csv"
summary_csv_path = "/content/drive/Shareddrives/v6_gax_outputs/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

# GAX Generation and Cheating Score Evaluation for ResNet34_v7

## GAX Generation

In [ ]:
# Execution for v7
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v7.pth --output_dir /content/drive/Shareddrives/v7_gax_outputs/outputs

Loading ResNet34 on cuda...

Processing Class: pneumonia | Target Label: 1
Found 602 images total. 298 remaining to process.
Generating Batched GAX (pneumonia): 100% 19/19 [08:16<00:00, 26.11s/it]

Processing Class: normal | Target Label: 0
Found 885 images total. 0 remaining to process.


## Contents Validation

In [ ]:
import os
# count number of contents for each drive to validate
#v7
# set folder path
folder_path = '/content/drive/Shareddrives/v7_gax_outputs/outputs'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/v7_gax_outputs/outputs

Deep Count (Includes all sub-folders):
Total Files:   2974
Total Folders: 0
Grand Total:   2974


## Cheating Score

In [ ]:
# running cheating score on v7 while its files are in session storage so no need to download
# Define paths to match your v7 execution
script_path = "/content/drive/Shareddrives/thesis/compute_cheating_score.py"
gax_directory = "/content/drive/Shareddrives/v7_gax_outputs/outputs"
output_csv_path = "/content/drive/Shareddrives/v7_gax_outputs/cheating_scores_v7.csv"


# Run the script
!python "{script_path}" \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v7"

Starting Cheating Score calculation for model: resnet34_v7...
Scoring normal: 100% 885/885 [1:38:12<00:00,  6.66s/it]
Scoring pneumonia: 100% 602/602 [38:00<00:00,  3.79s/it]

Finished! Results saved to /content/drive/Shareddrives/v7_gax_outputs/cheating_scores_v7.csv

--- Final Summary (resnet34_v7) ---
Total Images Scored: 1487
  - normal: 885 images
  - pneumonia: 602 images
Shortcut Learning Detected: 1330 images (89.4%)
  - normal: 762 images
  - pneumonia: 568 images
Average Model Cheating Score: 0.7200 (72.0%)
Summary metrics successfully saved to /content/drive/Shareddrives/v7_gax_outputs/summary_cheating_scores.csv


## Download Cheating Score

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/v7_gax_outputs/cheating_scores_v7.csv"
summary_csv_path = "/content/drive/Shareddrives/v7_gax_outputs/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

Found: cheating_scores_v7.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: summary_cheating_scores.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# GAX Generation and Cheating Score Evaluation for ResNet34_v8

## GAX Generation

In [ ]:
# Execution for v8
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v8.pth --output_dir /content/drive/Shareddrives/v8_gax_outputs/outputs

Loading ResNet34 on cuda...

Processing Class: pneumonia | Target Label: 1
Found 602 images total. 298 remaining to process.
Generating Batched GAX (pneumonia): 100% 19/19 [08:17<00:00, 26.16s/it]

Processing Class: normal | Target Label: 0
Found 885 images total. 0 remaining to process.


## Contents Validation

In [ ]:
# count number of contents for each drive to validate
#v8
# set folder path
import os
folder_path = '/content/drive/Shareddrives/v8_gax_outputs/outputs'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/v8_gax_outputs/outputs

Deep Count (Includes all sub-folders):
Total Files:   2974
Total Folders: 0
Grand Total:   2974


## Cheating Score

In [ ]:
# running cheating score on v8 while its files are in session storage so no need to download
# Define paths to match your v8 execution
script_path = "/content/drive/Shareddrives/thesis/compute_cheating_score.py"
gax_directory = "/content/drive/Shareddrives/v8_gax_outputs/outputs"
output_csv_path = "/content/drive/Shareddrives/v8_gax_outputs/cheating_scores_v8.csv"


# Run the script
!python "{script_path}" \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v8"

Starting Cheating Score calculation for model: resnet34_v8...
Scoring normal: 100% 885/885 [1:06:27<00:00,  4.51s/it]
Scoring pneumonia: 100% 602/602 [27:16<00:00,  2.72s/it]

Finished! Results saved to /content/drive/Shareddrives/v8_gax_outputs/cheating_scores_v8.csv

--- Final Summary (resnet34_v8) ---
Total Images Scored: 1487
  - normal: 885 images
  - pneumonia: 602 images
Shortcut Learning Detected: 1377 images (92.6%)
  - normal: 810 images
  - pneumonia: 567 images
Average Model Cheating Score: 0.7316 (73.2%)
Summary metrics successfully saved to /content/drive/Shareddrives/v8_gax_outputs/summary_cheating_scores.csv


## Download Cheating Score

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/v8_gax_outputs/cheating_scores_v8.csv"
summary_csv_path = "/content/drive/Shareddrives/v8_gax_outputs/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

Found: cheating_scores_v8.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: summary_cheating_scores.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# GAX Generation and Cheating Score Evaluation for ResNet34_v9

## GAX Generation

In [ ]:
# Execution for v9
!python generate_gax.py --batch_size 16 --model_path /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v9.pth --output_dir /content/drive/Shareddrives/v9_gax_outputs/outputs

Loading ResNet34 on cuda...

Processing Class: normal | Target Label: 0
Found 885 images total. 885 remaining to process.
Generating Batched GAX (normal): 100% 56/56 [22:23<00:00, 23.99s/it]

Processing Class: pneumonia | Target Label: 1
Found 602 images total. 602 remaining to process.
Generating Batched GAX (pneumonia): 100% 38/38 [15:54<00:00, 25.11s/it]


## Contents Validation

In [ ]:
# count number of contents for each drive to validate
#v9
# set folder path
folder_path = '/content/drive/Shareddrives/v9_gax_outputs/outputs'

print(f"Analyzing: {folder_path}\n")

# Counts absolutely everything, including items inside sub-folders.
file_count = 0
folder_count = 0

# os.walk goes through the directory tree top-down
for root, directories, files in os.walk(folder_path):
    folder_count += len(directories)
    file_count += len(files)

print("Deep Count (Includes all sub-folders):")
print(f"Total Files:   {file_count}")
print(f"Total Folders: {folder_count}")
print(f"Grand Total:   {file_count + folder_count}")

Analyzing: /content/drive/Shareddrives/v9_gax_outputs/outputs

Deep Count (Includes all sub-folders):
Total Files:   2974
Total Folders: 0
Grand Total:   2974


## Cheating Score

In [ ]:
# running cheating score on v9 while its files are in session storage so no need to download
# Define paths to match your v9 execution
gax_directory = "/content/drive/Shareddrives/v9_gax_outputs/outputs"
output_csv_path = "/content/drive/Shareddrives/v9_gax_outputs/cheating_scores_v9.csv"


# Run the script
!python compute_cheating_score.py \
    --gax_dir "{gax_directory}" \
    --output_csv "{output_csv_path}" \
    --model "resnet34_v9"

Starting Cheating Score calculation for model: resnet34_v9...
Scoring pneumonia: 100% 602/602 [23:45<00:00,  2.37s/it]
Scoring normal: 100% 885/885 [59:58<00:00,  4.07s/it]

Finished! Results saved to /content/drive/Shareddrives/v9_gax_outputs/cheating_scores_v9.csv

--- Final Summary (resnet34_v9) ---
Total Images Scored: 1487
  - normal: 885 images
  - pneumonia: 602 images
Shortcut Learning Detected: 1330 images (89.4%)
  - normal: 765 images
  - pneumonia: 565 images
Average Model Cheating Score: 0.7065 (70.6%)
Summary metrics successfully saved to /content/drive/Shareddrives/v9_gax_outputs/summary_cheating_scores.csv


## Download Cheating Score

In [ ]:
# download the csv file generated from the cheating score computation
from google.colab import files
import os

# Define the paths (must match the paths used in your processing script)
output_csv_path = "/content/drive/Shareddrives/v9_gax_outputs/cheating_scores_v9.csv"
summary_csv_path = "/content/drive/Shareddrives/v9_gax_outputs/summary_cheating_scores.csv"

def download_results():
    # Check for the detailed scores file
    if os.path.exists(output_csv_path):
        print(f"Found: {os.path.basename(output_csv_path)}. Initiating download...")
        files.download(output_csv_path)
    else:
        print(f"❌ Error: {output_csv_path} not found. Did the script finish successfully?")

    # Check for the summary file
    if os.path.exists(summary_csv_path):
        print(f"Found: {os.path.basename(summary_csv_path)}. Initiating download...")
        files.download(summary_csv_path)
    else:
        print(f"⚠️ Note: {summary_csv_path} not found.")

# Run the download function
download_results()

Found: cheating_scores_v9.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: summary_cheating_scores.csv. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>